In [1]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('health_data/authors.csv')
df_id_authors = df[['ID', 'Authors']].copy()

df_id_authors

,ID,Authors
0,119,"Bui Nam, Pham Nhat, Barnitz Jessica Jacqueline..."
1,120,"Röddiger Tobias, Beigl Michael, Hefenbrock Mic..."
2,121,"E. Nemati, S. Zhang, T. Ahmed, M. M. Rahman, J..."
3,122,"S. Zhang, E. Nemati, T. Ahmed, M. M. Rahman, J..."
4,123,"N. Nguyen, A. Chakma, N. Roy"
5,124,"A. Boukhayma, A. Barison, S. Haddad, A. Caizzone"
6,125,"S. -j. Jung, J. Ryu, W. Kim, S. Lee, J. Kim, H..."
7,126,"J. Meneses, O. Miranda, I. Sanchez, C. Álvarez..."
8,127,"J. Juez, D. Henao, F. Segura, R. Gómez, M. Le ..."
9,128,"K. -J. Kim, K. -T. Lim, J. w. Baek, M. Shin"


In [ ]:
# Build exact co-author matrix from comma-separated author strings
coauthor_matrix = pd.DataFrame(0, index=np.arange(1, len(df_id_authors)+1), columns=np.arange(1, len(df_id_authors)+1))
ids = df_id_authors['ID'].to_numpy(dtype=int)

def normalize_name(name: str) -> str:
    # lowercase + collapse internal whitespace
    return ' '.join(name.strip().lower().split())

def to_author_set(value) -> set:
    # Accept list or single comma-separated string
    if isinstance(value, list):
        names = value
    elif isinstance(value, str):
        # split on commas that separate authors
        names = [n for n in (x.strip() for x in value.split(',')) if n]
    else:
        names = []
    return {normalize_name(n) for n in names}

# Map ID -> normalized author set (robust to missing rows)
id_to_authors = {int(row['ID']): to_author_set(row['Authors']) for _, row in df_id_authors.iterrows()}

# Only connect papers sharing at least one EXACT author name (distance == 0)
for id_i in ids:
    authors_i = id_to_authors.get(id_i, set())
    for id_j in range(id_i + 1, len(ids) + 1):
        authors_j = id_to_authors.get(id_j, set())
        if authors_i and authors_j and authors_i.intersection(authors_j):
            coauthor_matrix.loc[id_i, id_j] = 1
            coauthor_matrix.loc[id_j, id_i] = 1  # symmetric

coauthor_matrix.to_csv('interconnections_datasets/coauthor_matrix_health.csv')
coauthor_matrix.head()

,1,2,3,4,5,6,7,8,9,10,...,41,42,43,44,45,46,47,48,49,50
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


: 